In [ ]:
from pathlib import Path

project_root = Path(".").absolute().parent.parent.parent
outputs_dir = project_root / "outputs"
outputs_dir

In [ ]:
run_folders = [
    f for f in outputs_dir.iterdir() if f.is_dir() and not f.name.startswith("TEST-")
]
run_folders.sort()

In [ ]:
from fusiontimeseries.experiments.generate_results_table import (
    parse_folder_name,
    read_rmse_data,
)

for folder in run_folders:
    method, context_len = parse_folder_name(folder.name)
    train_rmse, train_se = read_rmse_data(folder / "train_results.json")
    val_rmse, val_se = read_rmse_data(folder / "val_results.json")
    id_rmse, id_se = read_rmse_data(folder / "id_test_results.json")
    ood_rmse, ood_se = read_rmse_data(folder / "ood_test_results.json")
    batch9_rmse, batch9_se = read_rmse_data(folder / "batch9_test_results.json")
    break

In [ ]:
import numpy as np

from fusiontimeseries.experiments.config import FinetuningConfig
from fusiontimeseries.experiments.dataset import FluxDataset

In [ ]:
import matplotlib.pyplot as plt


subsampling = True
config = FinetuningConfig(
    eval_context_cutoff=1,
    train_context_cutoffs=[1],
    subsampling=subsampling,
    context_length=267 if subsampling else 800,
)

gyroswin_id_dataset = FluxDataset(
    namespaces=[
        # 'batch_1', 'batch_2', 'batch_3', 'batch_4', 'batch_5', 'batch_6', 'batch_7', 'batch_8',
        "batch_9",
        # 'batch_10',
        "gyroswin_id",
        "gyroswin_ood",
        "gyroswin_train",
        "gyroswin_val",
    ],
    config=config,
)  # Collect individual flux means for boxplot visualization
flux_boxplot_data = {}
# namespace -> cutoff -> list of individual means

for cutoff in [1, 11, 21, 41, 80]:
    for ns in gyroswin_id_dataset.namespaces:
        if ns not in flux_boxplot_data:
            flux_boxplot_data[ns] = {}

        flux_data = gyroswin_id_dataset.flux_data[ns]
        flux_context = [flux["energy_flux"][:cutoff] for flux in flux_data.values()]
        individual_flux_mean = np.mean(flux_context, axis=1)
        flux_boxplot_data[ns][cutoff] = individual_flux_mean.tolist()


# Prepare data for boxplot
context_lengths = [1, 11, 21, 41, 80]
namespaces = list(flux_boxplot_data.keys())
# namespaces = ["gyroswin_id", "gyroswin_ood", "gyroswin_val", "batch_9"]

fig, ax = plt.subplots(figsize=(12, 6))

# Create positions for boxplots
positions = []
data_to_plot = []
colors = plt.cm.tab10(np.linspace(0, 1, len(namespaces)))

for i, cutoff in enumerate(context_lengths):
    for j, ns in enumerate(namespaces):
        positions.append(i * (len(namespaces) + 1) + j)
        data_to_plot.append(flux_boxplot_data[ns][cutoff])

# Create boxplot
bp = ax.boxplot(
    data_to_plot, positions=positions, widths=0.6, patch_artist=True, showfliers=False
)

# Color boxes by namespace
for patch, pos in zip(bp["boxes"], positions):
    namespace_idx = pos % (len(namespaces) + 1)
    patch.set_facecolor(colors[namespace_idx])
    patch.set_alpha(0.7)

# Set x-axis labels
tick_positions = [
    i * (len(namespaces) + 1) + len(namespaces) // 2
    for i in range(len(context_lengths))
]
ax.set_xticks(tick_positions)
ax.set_xticklabels(context_lengths)

ax.set_xlabel("Context Length", fontsize=12)
ax.set_ylabel("Flux Value", fontsize=12)
ax.set_title("Flux Distribution by Context Length and Namespace", fontsize=14)
ax.grid(True, alpha=0.3, axis="y")

# Create legend
handles = [
    plt.Rectangle((0, 0), 1, 1, facecolor=colors[i], alpha=0.7)
    for i in range(len(namespaces))
]
ax.legend(handles, namespaces, loc="upper left", fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt


subsampling = False
config = FinetuningConfig(
    eval_context_cutoff=1,
    train_context_cutoffs=[1],
    subsampling=subsampling,
    context_length=267 if subsampling else 800,
)

gyroswin_id_dataset = FluxDataset(
    namespaces=[
        # 'batch_1', 'batch_2', 'batch_3', 'batch_4', 'batch_5', 'batch_6', 'batch_7', 'batch_8',
        "batch_9",
        # 'batch_10',
        "gyroswin_id",
        "gyroswin_ood",
        "gyroswin_train",
        "gyroswin_val",
    ],
    config=config,
)
# Collect individual flux means for boxplot visualization
flux_boxplot_data = {}
# namespace -> cutoff -> list of individual means

for cutoff in [1, 11, 21, 41, 80]:
    for ns in gyroswin_id_dataset.namespaces:
        if ns not in flux_boxplot_data:
            flux_boxplot_data[ns] = {}

        flux_data = gyroswin_id_dataset.flux_data[ns]
        flux_context = [flux["energy_flux"][:cutoff] for flux in flux_data.values()]
        individual_flux_mean = np.mean(flux_context, axis=1)
        flux_boxplot_data[ns][cutoff] = individual_flux_mean.tolist()


# Prepare data for boxplot
context_lengths = [1, 11, 21, 41, 80]
namespaces = list(flux_boxplot_data.keys())
# namespaces = ["gyroswin_id", "gyroswin_ood", "gyroswin_val", "batch_9"]

fig, ax = plt.subplots(figsize=(12, 6))

# Create positions for boxplots
positions = []
data_to_plot = []
colors = plt.cm.tab10(np.linspace(0, 1, len(namespaces)))

for i, cutoff in enumerate(context_lengths):
    for j, ns in enumerate(namespaces):
        positions.append(i * (len(namespaces) + 1) + j)
        data_to_plot.append(flux_boxplot_data[ns][cutoff])

# Create boxplot
bp = ax.boxplot(
    data_to_plot, positions=positions, widths=0.6, patch_artist=True, showfliers=False
)

# Color boxes by namespace
for patch, pos in zip(bp["boxes"], positions):
    namespace_idx = pos % (len(namespaces) + 1)
    patch.set_facecolor(colors[namespace_idx])
    patch.set_alpha(0.7)

# Set x-axis labels
tick_positions = [
    i * (len(namespaces) + 1) + len(namespaces) // 2
    for i in range(len(context_lengths))
]
ax.set_xticks(tick_positions)
ax.set_xticklabels(context_lengths)

ax.set_xlabel("Context Length", fontsize=12)
ax.set_ylabel("Flux Value", fontsize=12)
ax.set_title("Flux Distribution by Context Length and Namespace", fontsize=14)
ax.grid(True, alpha=0.3, axis="y")

# Create legend
handles = [
    plt.Rectangle((0, 0), 1, 1, facecolor=colors[i], alpha=0.7)
    for i in range(len(namespaces))
]
ax.legend(handles, namespaces, loc="upper left", fontsize=8, ncol=2)

plt.tight_layout()
plt.show()